# 📘 Colab Notebook: Evaluate LLM Responses from a Local JSON File Using Llumo

## 📝 Notebook Overview
This notebook demonstrates how to upload and read a local JSON or JSONL file containing LLM interaction logs, format the data, and then evaluate the model's responses using Llumo’s powerful input-level metrics.

### ✨ Metrics included:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
- ▶ Hallucination
- 🛠️ Context Utilization
  
---

## 🚀 What you will do in this notebook:
- 📤 Upload a local JSON or JSONL file containing your evaluation data.  
- 🔄 Format the raw data from the file into the standardized structure required by Llumo.
- 🤖 Evaluate the model's output for correctness, completeness, bias, harmfulness, and more.
- 📊 View the detailed evaluation results in a structured table.  
---

### **⚙️ 1. Install Dependencies**
First, we'll install the necessary Python libraries.
- `llumo`: The official SDK for the Llumo platform.
- `pandas`: Used for data manipulation and displaying results.

In [ ]:
!pip install llumo pandas -q

### **📚 2. Import Required Libraries**

In [ ]:
import os
import pandas as pd
import json
import getpass
from llumo import LlumoClient
from google.colab import files

### **🔑 3. Configure API Key & Upload Your Data File**

This step will prompt you for your Llumo API key and then ask you to upload your data file.

1.  **Llumo API Key**: Get your key from the [Llumo Dashboard](https://llumo.ai/dashboard).
2.  **Data File**: You can upload either a standard JSON file (containing a list of objects) or a JSON Lines file (`.jsonl`), where each line is a separate JSON object.

In [ ]:
# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = getpass.getpass("Enter Your Llumo API Key: ")
llumo_key = os.getenv("LLUMO_API_KEY")

# --- File Upload ---
print("Please upload your JSON or JSONL data file.")
uploaded = files.upload()

if not uploaded:
    print("\nNo file uploaded. Please run the cell again and select a file to continue.")
else:
    # Get the name of the uploaded file
    file_name = next(iter(uploaded))
    print(f"\nUploaded '{file_name}' successfully.")

### **📄 4. Read Data from JSON File**

This step reads the content of your uploaded file. It automatically detects whether it's a standard JSON or a JSONL file and parses it into a list of dictionaries.

In [ ]:
raw_logs = []

if 'file_name' in locals() and file_name:
    try:
        # Check if the file is JSONL
        if file_name.endswith('.jsonl'):
            print("Detected JSONL file. Reading line by line.")
            with open(file_name, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip(): # ensure the line is not empty
                        raw_logs.append(json.loads(line))
        # Assume standard JSON otherwise
        else:
            print("Detected JSON file. Reading entire file.")
            with open(file_name, 'r', encoding='utf-8') as f:
                raw_logs = json.load(f)
        
        print(f"Successfully loaded {len(raw_logs)} records from '{file_name}'.")

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from file: {e}")
        print("Please ensure the file is a valid JSON or JSONL file.")
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
else:
    print("File not found. Please upload a file in the previous step.")

# Preview the first raw log to understand its structure
if raw_logs:
    print("\nSample raw log from the file:")
    # Make sure the log is a dict before printing
    if isinstance(raw_logs[0], dict):
      print(json.dumps(raw_logs[0], indent=2))
    else:
      print(raw_logs[0])

### **🔄 5. Format Data for Llumo Evaluation**
Llumo's `evaluateMultiple` function expects a list of dictionaries, where each dictionary contains specific keys like `query`, `context`, and `output`. 

The function below converts your raw data into this standardized format. **You must adjust the key mappings** inside the function to match the keys in your JSON file.

In [ ]:
def convert_to_llumo_format(logs):
  """
  Converts a list of raw log dictionaries from a JSON file into the format required by Llumo.
  
  Args:
    logs (list): A list of dictionaries, where each dictionary is an object from the JSON file.
    
  Returns:
    list: A list of formatted dictionaries for Llumo evaluation.
  """
  formatted_data = []
  for log in logs:
    # ➡️ TODO: Adjust these key names to match your JSON object's structure.
    # For example, if your user's prompt is a key called 'question',
    # change 'userPrompt' to 'question'.
    formatted_dict = {
        'query': log.get('userPrompt', ''),        # Map your key for the user's question/prompt
        'context': log.get('retrievedContext', ''),  # Map your key for the retrieved context
        'output': log.get('modelResponse', ''),     # Map your key for the model's generated response
        # Optional: Map the ground truth key if you have one
        'ground_truth': log.get('referenceAnswer', None) 
    }
    formatted_data.append(formatted_dict)
  return formatted_data

# Process the loaded logs
if raw_logs:
    evaluation_data = convert_to_llumo_format(raw_logs)
    
    # Preview the first formatted item to verify the mapping
    print("Sample log after formatting for Llumo:")
    print(json.dumps(evaluation_data[0], indent=2))
else:
    evaluation_data = []
    print("No data to format.")

### **🤖 6. Initialize Llumo Client and Evaluate Responses**

With our data loaded and formatted, we can now proceed with the evaluation. We will initialize the `LlumoClient` and call the `evaluateMultiple` function.

We pass our formatted data and select the KPIs we want to measure:
- 🎯 **Response Correctness**: Is the answer factually accurate based on the context?
- 🧩 **Response Completeness**: Does the answer fully address the user's query?
- 🧠 **Response Bias**: Is the response free from demographic or social biases?
- ☣️ **Harmfulness**: Does the response contain toxic, hateful, or unsafe content?
- 🛠️ **Context Utilization**: How well does the answer use the provided context?
- ▶ **Hallucination**: Does the answer invent information not present in the context?

In [ ]:
evalDf = pd.DataFrame()

if evaluation_data and llumo_key:
    # Initialize the LlumoClient with your API key
    client = LlumoClient(api_key = llumo_key)

    # Call the evaluation function
    print("Starting evaluation with Llumo...")
    evalDf = client.evaluateMultiple(
      data = evaluation_data,  # The formatted data from the previous step
      evals = ["Response Completeness", "Response Correctness", "Response Bias", "Context Utilization", "Hallucination"], # Selected evaluation KPIs
      getDataFrame = True # Return result as a pandas DataFrame
    )
    print("Evaluation complete!")
else:
    print("Skipping evaluation. Ensure data was loaded from your file and the Llumo API key is set.")

### **📊 7. View Evaluation Results**
The results are returned in a pandas DataFrame, providing a detailed breakdown of each metric for every record. This allows for easy analysis, sorting, and filtering to identify problematic responses and gain insights into your model's performance.

In [ ]:
# Display the full evaluation results table
if not evalDf.empty:
    # Configure pandas to display wide columns for better readability
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 80)
    display(evalDf)
else:
    print("No evaluation results to display.")